<a href="https://colab.research.google.com/github/KamilAkarsu/DoktoraDeneyleri/blob/main/Exp_1_2_Dinamik_Veri_Y%C3%BCkleyici_ve_Augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import zipfile
import shutil
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# 1. Drive'ı Bağlama
from google.colab import drive
drive.mount('/content/drive')

# 2. İşlenmiş Veriyi Colab Belleğine Alma
DRIVE_ZIP_YOLU = '/content/drive/MyDrive/Doktora/Verisetleri_tik/Islenmis_MURA.zip'
YEREL_VERI_KLASORU = '/content/dataset'

# Klasör yoksa veya boşsa zip'ten çıkar
if not os.path.exists(YEREL_VERI_KLASORU) or len(os.listdir(YEREL_VERI_KLASORU)) == 0:
    print("Veri seti Colab diskine çıkartılıyor...")
    if not os.path.exists(YEREL_VERI_KLASORU):
        os.makedirs(YEREL_VERI_KLASORU)
    with zipfile.ZipFile(DRIVE_ZIP_YOLU, 'r') as zip_ref:
        zip_ref.extractall(YEREL_VERI_KLASORU)
    print("Çıkartma işlemi tamamlandı.")
else:
    print("Veri seti zaten mevcut, çıkartma işlemi atlandı.")

# Klasör içeriğini kontrol et (Hata ayıklama için)
print(f"'{YEREL_VERI_KLASORU}' ana dizin içeriği: {os.listdir(YEREL_VERI_KLASORU)}")

# Doğru veri yolu (path) bulma mantığı
possible_train_path = os.path.join(YEREL_VERI_KLASORU, 'train')

if not os.path.exists(possible_train_path):
    found = False
    for item in os.listdir(YEREL_VERI_KLASORU):
        sub_path = os.path.join(YEREL_VERI_KLASORU, item)
        if os.path.isdir(sub_path) and 'train' in os.listdir(sub_path):
            YEREL_VERI_KLASORU = sub_path
            print(f"Veri seti yolu güncellendi: {YEREL_VERI_KLASORU}")
            found = True
            break

    if not found:
        raise FileNotFoundError(f"'train' klasörü '{YEREL_VERI_KLASORU}' altında veya alt klasörlerinde bulunamadı.")

# 3. PyTorch Dataset Sınıfının Tanımlanması (Recursive Arama Eklendi)
class KemikKirigiDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # Binary Classification (İkili Sınıflandırma) Hedefi: 0=Normal, 1=Kırık
        self.classes = ['0_Normal', '1_Fracture']

        # Alt dizinlerdeki tüm dosyaları tarama
        for root, dirs, files in os.walk(root_dir):
            for img_name in files:
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    tam_yol = os.path.join(root, img_name)

                    # MURA standardı: yolda 'positive' geçiyorsa kırık (1), 'negative' geçiyorsa normal (0)
                    if 'positive' in tam_yol.lower():
                        label = 1
                    elif 'negative' in tam_yol.lower():
                        label = 0
                    else:
                        continue # Etiketi belirsiz olanları atla

                    self.image_paths.append(tam_yol)
                    self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# 4. Dinamik Veri Artırma
train_transforms = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Yolları Güncelle
TRAIN_DIR = os.path.join(YEREL_VERI_KLASORU, 'train')
VAL_DIR = os.path.join(YEREL_VERI_KLASORU, 'valid')

print(f"Eğitim klasörü: {TRAIN_DIR}")
print(f"Doğrulama klasörü: {VAL_DIR}")

# Dataset nesnelerini oluşturma
train_dataset = KemikKirigiDataset(root_dir=TRAIN_DIR, transform=train_transforms)
val_dataset = KemikKirigiDataset(root_dir=VAL_DIR, transform=test_transforms)

# 5. DataLoader Tanımlamaları
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Eğitim Seti Boyutu: {len(train_dataset)} görüntü")
print(f"Doğrulama Seti Boyutu: {len(val_dataset)} görüntü")

# 6. MixUp Artırma Fonksiyonu
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).cuda()

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Veri seti zaten mevcut, çıkartma işlemi atlandı.
'/content/dataset' ana dizin içeriği: ['MURA-v1.1']
Veri seti yolu güncellendi: /content/dataset/MURA-v1.1
Eğitim klasörü: /content/dataset/MURA-v1.1/train
Doğrulama klasörü: /content/dataset/MURA-v1.1/valid
Eğitim Seti Boyutu: 36808 görüntü
Doğrulama Seti Boyutu: 3197 görüntü
